In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv('IPL.csv')

print(df.shape)
df.head()

In [ ]:
print(df.columns)

In [ ]:
df.columns = df.columns.str.strip()


df = df.dropna(subset=['runs_total'])

In [ ]:
#powerplay runs
powerplay = df[df['over'] <= 6]

pp_runs = powerplay.groupby(['match_id','innings','batting_team'])['runs_total'].sum().reset_index()
pp_runs.rename(columns={'runs_total':'powerplay_runs'}, inplace=True)

In [ ]:
#death overs
death = df[df['over'] >= 16]

death_runs = death.groupby(['match_id','innings','batting_team'])['runs_total'].sum().reset_index()
death_runs.rename(columns={'runs_total':'death_runs'}, inplace=True)

In [ ]:
total_runs = df.groupby(['match_id','innings','batting_team'])['runs_total'].sum().reset_index()

In [ ]:
team_stats = total_runs.merge(pp_runs, on=['match_id','innings','batting_team'])
team_stats = team_stats.merge(death_runs, on=['match_id','innings','batting_team'])

team_stats.head()

In [ ]:
# Pivot scores
match_scores = total_runs.pivot_table(
    index='match_id',
    columns='innings',
    values='runs_total'
).reset_index()

# Keep only innings 1 and 2 (ignore super overs)
match_scores = match_scores[[col for col in match_scores.columns if col in ['match_id', 1, 2]]]

# Rename properly
match_scores.columns = ['match_id', 'innings1_runs', 'innings2_runs']

In [ ]:
print(match_scores.head())

In [ ]:
match_scores['winner_innings'] = np.where(
    match_scores['innings1_runs'] > match_scores['innings2_runs'], 1, 2
)

In [ ]:
team_stats = team_stats.merge(match_scores, on='match_id')

team_stats['win'] = np.where(
    team_stats['innings'] == team_stats['winner_innings'], 1, 0
)

team_stats.head()

In [ ]:
#impact of powerplay
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x='win', y='powerplay_runs', data=team_stats)
plt.title("Powerplay Runs vs Winning")
plt.show()

In [ ]:
#ipact of deathovers
sns.boxplot(x='win', y='death_runs', data=team_stats)
plt.title("Death Overs Runs vs Winning")
plt.show()

In [ ]:
sns.boxplot(x='win', y='runs_total', data=team_stats)
plt.title("Total Runs vs Winning")
plt.show()

no single factor affects the outcome of the game


In [ ]:
#heatmap
corr = team_stats[['powerplay_runs','death_runs','runs_total','win']].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title(" Correlation")
plt.show()

In [ ]:
sns.scatterplot(
    x='powerplay_runs',
    y='death_runs',
    hue='win',
    data=team_stats
)
plt.title("Powerplay vs Death Runs (Win/Loss)")
plt.show()

In [ ]:
team_stats.describe()